In [3]:
import os
import sys

root = "/tmp2/maitanha/vgu/cll_vlm/cll_vlm"

if root not in sys.path:
    sys.path.append(root)

import torch
import re
import csv
import random
import numpy as np
import pandas as pd
import yaml
import json
from pathlib import Path
from collections import Counter
from IPython.display import display
from argparse import ArgumentParser
from torch.utils.data import DataLoader
from tqdm import tqdm
import pdb
from dataset.cifar10 import CIFAR10Dataset
from dataset.cifar20 import CIFAR20Dataset, CIFAR100Dataset
from dataset.tiny200 import Tiny200Dataset

from models.llava_classifier import LLaVAClassifier
from models.qwen_classifier import QWENClassifier
from models.clip_model import CLIPModel
from PIL import Image   

/tmp2/maitanha/vgu/cll_vlm/venv_cll_qwen/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python version is above 3.10, patching the collections module.


/tmp2/maitanha/vgu/cll_vlm/venv_cll_qwen/lib/python3.13/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


In [4]:
def load_config(config_path):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file '{config_path}' not found.")
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def collate_fn(batch):
    images, labels = zip(*batch)
    return list(images), list(labels)

def load_dataset(data_name):
    DATA_DIR = "/tmp2/maitanha/vgu/cll_vlm/cll_vlm/data"
    data_root_path = os.path.join(DATA_DIR, data_name)

    if data_name == "cifar10":
        dataset = CIFAR10Dataset(
            root=data_root_path,
            train=True,
            transform=None
        )
    elif data_name == "cifar20":
        dataset = CIFAR20Dataset(
            root=data_root_path,
            train=True,
            transform=None
        )
    elif data_name == "cifar100":
        dataset = CIFAR100Dataset(
            root = data_root_path,
            train=True,
            transform=None
        )
        fine_classes_raw = dataset.get_fine_classes()
        fine_classes = [
            CIFAR100Dataset.preprocess_label(lbl)
            for lbl in fine_classes_raw
        ]
        coarse_classes = dataset.get_coarse_classes()
    elif data_name == "tiny200/tiny-imagenet-200":
        dataset = Tiny200Dataset(
            root=data_root_path,
            train=True,
            transform=None
        )
    else:
        raise ValueError(f"Dataset '{data_name}' chưa được hỗ trợ trong hàm load_dataset.")
    
    return dataset

In [5]:
dataset = load_dataset("cifar100")
fine_classes_raw = dataset.get_fine_classes()
fine_classes = [
    CIFAR100Dataset.preprocess_label(lbl)
    for lbl in fine_classes_raw
]
coarse_classes = dataset.get_coarse_classes()

original_dataset, shuffled_dataset = dataset.get_shuffled_labels_dataset(seed=42)

# Analyze Results

In [10]:
def analyze_result(json_path):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        return {
            "error": str(e),
            "file": Path(json_path).name
        }

    total = len(data)
    correct_candidate = 0
    correct_final = 0
    empty_answer = 0
    
    for item in data:
        true_label = str(item.get('true_label', '')).strip().lower()
        candidate = item.get('clip_topk_candidates', [])
        answer = item.get('answer', [])

        if isinstance(candidate, str): 
            candidate = [candidate]
        elif candidate is None:
            candidate = []
        else:
            candidate = [str(c).strip().lower() for c in candidate]

        if isinstance(answer, str): answer = [answer]
        candidate = [str(c).strip().lower() for c in candidate]
        answer = [str(a).strip().lower() for a in answer]

        if not answer or (len(answer) == 1 and answer[0] == ""): # None or empty
            empty_answer += 1
            continue

        if true_label in candidate:
            correct_candidate += 1
        if true_label in answer:
            correct_final += 1

    return {
        'file': Path(json_path).name,
        'total': total,
        'c_count': correct_candidate,
        'f_count': correct_final,
        'candidate_acc': (correct_candidate / total) * 100 if total > 0 else 0,
        'final_acc': (correct_final / total) * 100 if total > 0 else 0,
        'missed': (correct_candidate - correct_final),
        'NO/Empty ans': empty_answer,
        'error': None
    }

from pathlib import Path
import pandas as pd
import json

# --- THỰC THI VÀ HIỂN THỊ ---
JSON_DIR = Path("/tmp2/maitanha/vgu/cll_vlm/cll_vlm/results/baseline3/baseline3_1/cifar100")
json_files = sorted(JSON_DIR.glob("*.json"))

summary_list = []
all_results = []

for jf in json_files:
    res = analyze_result(jf)

    if res.get('error'):
        print(f"⚠️ SKIP: {res['file']} - {res['error']}")
        continue

    all_results.append(res)

    summary_list.append({
        'File Name': res['file'],
        'Total': f"{res['total']:,}",
        'Cand Correct': f"{res['c_count']:,}",
        'Cand Acc (%)': f"{res['candidate_acc']:.2f}%",
        'Final Correct': f"{res['f_count']:,}",
        'Final Acc (%)': f"{res['final_acc']:.2f}%",
        'Missed (Cand-Final)': f"{res['missed']:,}",
        'Empty Ans': res['NO/Empty ans']
    })

# --- BẢNG TỔNG HỢP ---
df_summary = pd.DataFrame(summary_list)

print("\n" + "="*50)
print("📊 SUMMARIZE BASELINE CLIP->VLM")
print("="*50)
display(df_summary)



📊 SUMMARIZE BASELINE CLIP->VLM


,File Name,Total,Cand Correct,Cand Acc (%),Final Correct,Final Acc (%),Missed (Cand-Final),Empty Ans
0,baseline3_1_vitl14_qwen3_4b_cifar100_topk1.json,"50,000","37,986",75.97%,"37,007",74.01%,979,0
1,baseline3_1_vitl14_qwen3_4b_cifar100_topk10.json,"50,000","48,224",96.45%,"40,745",81.49%,"7,479",0
2,baseline3_1_vitl14_qwen3_4b_cifar100_topk20.json,"50,000","49,104",98.21%,"40,802",81.60%,"8,302",0
3,baseline3_1_vitl14_qwen3_4b_cifar100_topk3.json,"50,000","45,019",90.04%,"39,930",79.86%,"5,089",0
4,baseline3_1_vitl14_qwen3_4b_cifar100_topk5.json,"50,000","46,744",93.49%,"40,482",80.96%,"6,262",0
5,baseline3_1_vitl14_qwen3_8b_cifar100_topk1.json,"50,000","37,986",75.97%,"36,015",72.03%,"1,971",0
6,baseline3_1_vitl14_qwen3_8b_cifar100_topk10.json,"50,000","48,224",96.45%,"40,512",81.02%,"7,712",0
7,baseline3_1_vitl14_qwen3_8b_cifar100_topk20.json,"50,000","49,104",98.21%,"40,601",81.20%,"8,503",0
8,baseline3_1_vitl14_qwen3_8b_cifar100_topk3.json,"50,000","45,019",90.04%,"39,913",79.83%,"5,106",0
9,baseline3_1_vitl14_qwen3_8b_cifar100_topk5.json,"50,000","46,744",93.49%,"40,322",80.64%,"6,422",0


### Tiny200

In [7]:
dataset = load_dataset("tiny200/tiny-imagenet-200")
fine_classes = list(dataset.classes)

original_dataset, shuffled_dataset = dataset.get_shuffled_labels_dataset(seed=42)

In [8]:
fine_classes

['Egyptian cat',
 'reel',
 'volleyball',
 'rocking chair',
 'lemon',
 'bullfrog',
 'basketball',
 'cliff',
 'espresso',
 'plunger',
 'parking meter',
 'German shepherd',
 'dining table',
 'monarch',
 'brown bear',
 'school bus',
 'pizza',
 'guinea pig',
 'umbrella',
 'organ',
 'oboe',
 'maypole',
 'goldfish',
 'potpie',
 'hourglass',
 'seashore',
 'computer keyboard',
 'Arabian camel',
 'ice cream',
 'nail',
 'space heater',
 'cardigan',
 'baboon',
 'snail',
 'coral reef',
 'albatross',
 'spider web',
 'sea cucumber',
 'backpack',
 'Labrador retriever',
 'pretzel',
 'king penguin',
 'sulphur butterfly',
 'tarantula',
 'lesser panda',
 'pop bottle',
 'banana',
 'sock',
 'cockroach',
 'projectile',
 'beer bottle',
 'mantis',
 'freight car',
 'guacamole',
 'remote control',
 'European fire salamander',
 'lakeside',
 'chimpanzee',
 'pay-phone',
 'fur coat',
 'alp',
 'lampshade',
 'torch',
 'abacus',
 'moving van',
 'barrel',
 'tabby',
 'goose',
 'koala',
 'bullet train',
 'CD player',
 'te

In [11]:
from pathlib import Path
import pandas as pd
import json

# --- THỰC THI VÀ HIỂN THỊ ---
JSON_DIR = Path("/tmp2/maitanha/vgu/cll_vlm/cll_vlm/results/baseline3/baseline3_1/tiny200")
json_files = sorted(JSON_DIR.glob("*.json"))

summary_list = []
all_results = []

for jf in json_files:
    res = analyze_result(jf)

    if res.get('error'):
        print(f"⚠️ SKIP: {res['file']} - {res['error']}")
        continue

    all_results.append(res)

    summary_list.append({
        'File Name': res['file'],
        'Total': f"{res['total']:,}",
        'Cand Correct': f"{res['c_count']:,}",
        'Cand Acc (%)': f"{res['candidate_acc']:.2f}%",
        'Final Correct': f"{res['f_count']:,}",
        'Final Acc (%)': f"{res['final_acc']:.2f}%",
        'Missed (Cand-Final)': f"{res['missed']:,}",
        'Empty Ans': res['NO/Empty ans']
    })

# --- BẢNG TỔNG HỢP ---
df_summary = pd.DataFrame(summary_list)

print("\n" + "="*50)
print("📊 SUMMARIZE BASELINE CLIP->VLM")
print("="*50)
display(df_summary)



📊 SUMMARIZE BASELINE CLIP->VLM


,File Name,Total,Cand Correct,Cand Acc (%),Final Correct,Final Acc (%),Missed (Cand-Final),Empty Ans
0,baseline3_1_vitl14_qwen3_4b_tiny200_topk1.json,"100,000","73,028",73.03%,"66,059",66.06%,"6,969",0
1,baseline3_1_vitl14_qwen3_4b_tiny200_topk10.json,"100,000","93,640",93.64%,"75,025",75.02%,"18,615",0
2,baseline3_1_vitl14_qwen3_4b_tiny200_topk20.json,"100,000","96,248",96.25%,"74,964",74.96%,"21,284",0
3,baseline3_1_vitl14_qwen3_4b_tiny200_topk3.json,"100,000","86,312",86.31%,"73,201",73.20%,"13,111",0
4,baseline3_1_vitl14_qwen3_4b_tiny200_topk5.json,"100,000","90,043",90.04%,"74,858",74.86%,"15,185",0
